Image → MediaPipe → 21 landmarks → CSV file

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [ ]:
import pandas as pd
df = pd.read_csv("data.csv")

In [2]:
df.head()

,0.30983632802963257,0.33478423953056335,0.3416069746017456,0.32744109630584717,0.38675516843795776,0.29366809129714966,0.4496811032295227,0.2790496349334717,0.5069965124130249,0.2807992696762085,...,0.16637016832828522,0.38836485147476196,0.061558835208415985,0.5423290133476257,0.08528617024421692,0.6207449436187744,0.12856897711753845,0.6702990531921387,0.1621728390455246,0
0,0.295349,0.284124,0.367255,0.234559,0.427255,0.189044,0.508409,0.186305,0.578500,0.203573,...,0.162363,0.350890,0.094866,0.487371,0.095268,0.563145,0.126531,0.619142,0.152296,0
1,0.353220,0.306206,0.375594,0.333976,0.394961,0.341689,0.410915,0.362555,0.398483,0.400742,...,0.228093,0.515288,0.031910,0.646984,0.080061,0.702229,0.145932,0.734558,0.197283,0
2,0.451463,0.667722,0.589202,0.586073,0.661007,0.445313,0.726390,0.326788,0.728625,0.190804,...,0.185340,0.292186,0.342415,0.383435,0.175175,0.524070,0.170965,0.612457,0.222571,0
3,0.411374,0.810098,0.591768,0.621822,0.630915,0.452519,0.695690,0.334001,0.744391,0.241029,...,0.173036,0.380459,0.419378,0.516867,0.186361,0.660568,0.181118,0.741655,0.231109,0
4,0.370802,0.821684,0.569486,0.619552,0.623617,0.444245,0.696072,0.331015,0.754847,0.246035,...,0.183672,0.362140,0.414848,0.518685,0.194411,0.665102,0.184698,0.753946,0.228628,0


In [3]:
df.shape

(1622, 43)

In [4]:
columns = []
for i in range(21):
    columns.extend([f"x{i}", f"y{i}"])
columns.append("label")
df.columns = columns

In [11]:
print(f"Original shape: {df.shape}")

Original shape: (1622, 43)


In [14]:
# ========================= CLEAN RARE CLASSES =========================
print("\nClass distribution BEFORE cleaning:")
print(df['label'].value_counts())

min_samples = 5
class_counts = df['label'].value_counts()
rare_classes = class_counts[class_counts < min_samples].index.tolist()

if rare_classes:
    print(f"\n⚠️ Removing {len(rare_classes)} rare classes (< {min_samples} samples):")
    print(rare_classes)
    df = df[~df['label'].isin(rare_classes)]
    print(f"✅ New shape after removal: {df.shape}")
else:
    print("✅ All classes already have enough samples!")

print("\nClass distribution AFTER cleaning:")
print(df['label'].value_counts())



Class distribution BEFORE cleaning:
label
5    70
9    70
4    69
3    68
f    68
l    68
k    67
d    66
7    65
b    65
u    63
8    61
1    60
g    54
z    54
i    53
r    52
v    52
6    51
2    49
w    49
p    48
x    47
h    46
y    35
c    33
0    28
e    23
a    15
o    15
j    13
q    13
s    12
n    11
m     8
t     1
Name: count, dtype: int64

⚠️ Removing 1 rare classes (< 5 samples):
['t']
✅ New shape after removal: (1621, 43)

Class distribution AFTER cleaning:
label
5    70
9    70
4    69
3    68
f    68
l    68
k    67
d    66
7    65
b    65
u    63
8    61
1    60
g    54
z    54
i    53
r    52
v    52
6    51
2    49
w    49
p    48
x    47
h    46
y    35
c    33
0    28
e    23
a    15
o    15
j    13
q    13
s    12
n    11
m     8
Name: count, dtype: int64


In [15]:
#prepare data 
X = df.drop("label", axis =  1)
y = df["label"]

In [16]:
#encode label (required for clean matrices)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [17]:
print(y.value_counts())

label
5    70
9    70
4    69
3    68
f    68
l    68
k    67
d    66
7    65
b    65
u    63
8    61
1    60
g    54
z    54
i    53
r    52
v    52
6    51
2    49
w    49
p    48
x    47
h    46
y    35
c    33
0    28
e    23
a    15
o    15
j    13
q    13
s    12
n    11
m     8
Name: count, dtype: int64


In [18]:
# Train-test split (stratified = balanced classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, stratify=y_encoded, random_state=42
)

In [19]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [20]:
# ========================= TRAIN =========================
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
# ========================= RESULTS =========================
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc  = accuracy_score(y_test,  model.predict(X_test))

In [23]:
print("\n" + "="*60)
print("STAGE 1 RESULTS (After Cleaning)")
print("="*60)
print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test  Accuracy : {test_acc:.4f}")
print(f"Gap            : {train_acc - test_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, model.predict(X_test), target_names=le.classes_))

# Save everything
joblib.dump(model, "model_stage1.pkl")
joblib.dump(scaler, "scaler_stage1.pkl")
joblib.dump(le, "label_encoder_stage1.pkl")
print("\n✅ Model saved successfully!")


STAGE 1 RESULTS (After Cleaning)
Train Accuracy : 1.0000
Test  Accuracy : 0.9662
Gap            : 0.0338

Classification Report:
              precision    recall  f1-score   support

           0       0.86      1.00      0.92         6
           1       0.92      1.00      0.96        12
           2       0.82      0.90      0.86        10
           3       1.00      1.00      1.00        14
           4       0.93      1.00      0.97        14
           5       1.00      0.93      0.96        14
           6       1.00      0.90      0.95        10
           7       0.93      1.00      0.96        13
           8       1.00      0.92      0.96        12
           9       1.00      1.00      1.00        14
           a       0.75      1.00      0.86         3
           b       1.00      1.00      1.00        13
           c       1.00      1.00      1.00         7
           d       1.00      1.00      1.00        13
           e       0.83      1.00      0.91         5
     

    SECOND STAGE    